In [ ]:
# ─── Experiment 2, Step 1: Rebuild Per-Video Frame Sequences ────────────────
# Reconstructs ordered (video_id -> frames) sequences from the split folders
# already created in cell 7. Windows are built within a single split only,
# so no video ever crosses train/val/test — same leakage protection as before.

import re
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as tv_models
import torchvision.transforms as T
from PIL import Image

WINDOW_SIZE = 12     # frames per sequence (~12 sec at your 1 fps extraction)
STRIDE = 6            # step between windows (50% overlap)

FRAME_IDX_RE = re.compile(r'_frame_(\d+)')

def extract_frame_idx(filename):
    m = FRAME_IDX_RE.search(filename)
    return int(m.group(1)) if m else 0

def build_video_sequences(split_dir, class_names):
    """
    Scans YOLO_DIR/<split>/<class>/*.jpg and groups them back into
    ordered per-video sequences, then slices into fixed-length windows.
    Returns a list of dicts: {video_id, class_name, frame_paths (ordered)}
    """
    video_frames = {}  # video_id -> list of (frame_idx, path, class_name)

    for class_name in class_names:
        class_dir = split_dir / class_name
        if not class_dir.exists():
            continue
        for img_path in class_dir.glob('*.jpg'):
            video_id = img_path.stem.rsplit('_frame_', 1)[0]
            frame_idx = extract_frame_idx(img_path.name)
            video_frames.setdefault(video_id, []).append((frame_idx, img_path, class_name))

    windows = []
    for video_id, frames in video_frames.items():
        frames.sort(key=lambda x: x[0])  # restore temporal order
        class_name = frames[0][2]        # single segment label per video
        paths = [f[1] for f in frames]

        for start in range(0, max(1, len(paths) - WINDOW_SIZE + 1), STRIDE):
            window_paths = paths[start:start + WINDOW_SIZE]
            if len(window_paths) < WINDOW_SIZE:
                continue  # drop short trailing windows
            windows.append({
                'video_id': video_id,
                'class_name': class_name,
                'frame_paths': window_paths,
            })

    return windows

train_windows = build_video_sequences(YOLO_DIR / 'train', CLASS_NAMES)
val_windows   = build_video_sequences(YOLO_DIR / 'val', CLASS_NAMES)
test_windows  = build_video_sequences(YOLO_DIR / 'test', CLASS_NAMES)

print(f"✅ Train windows: {len(train_windows)}")
print(f"✅ Val windows:   {len(val_windows)}")
print(f"✅ Test windows:  {len(test_windows)}")

In [ ]:
# ─── Experiment 2, Step 2: Sequence Dataset ─────────────────────────────────

IMG_SIZE = 224

seq_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class SegmentSequenceDataset(Dataset):
    def __init__(self, windows, class_to_id, transform):
        self.windows = windows
        self.class_to_id = class_to_id
        self.transform = transform

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        w = self.windows[idx]
        imgs = []
        for p in w['frame_paths']:
            img = Image.open(p).convert('RGB')
            imgs.append(self.transform(img))
        seq_tensor = torch.stack(imgs, dim=0)  # (WINDOW_SIZE, C, H, W)
        label = self.class_to_id[w['class_name']]
        return seq_tensor, label

train_seq_ds = SegmentSequenceDataset(train_windows, CLASS_TO_ID, seq_transform)
val_seq_ds   = SegmentSequenceDataset(val_windows, CLASS_TO_ID, seq_transform)
test_seq_ds  = SegmentSequenceDataset(test_windows, CLASS_TO_ID, seq_transform)

BATCH_SIZE_SEQ = 4  # sequences are memory-heavy (BATCH x WINDOW frames each) — start small

train_seq_loader = DataLoader(train_seq_ds, batch_size=BATCH_SIZE_SEQ, shuffle=True, num_workers=2)
val_seq_loader   = DataLoader(val_seq_ds, batch_size=BATCH_SIZE_SEQ, shuffle=False, num_workers=2)
test_seq_loader  = DataLoader(test_seq_ds, batch_size=BATCH_SIZE_SEQ, shuffle=False, num_workers=2)

print(f"✅ Train batches: {len(train_seq_loader)} | Val: {len(val_seq_loader)} | Test: {len(test_seq_loader)}")

In [ ]:
# ─── Experiment 2, Step 3: CNN Backbone + LSTM Sequence Classifier ──────────

class CNNSequenceClassifier(nn.Module):
    def __init__(self, num_classes, hidden_size=256, lstm_layers=1,
                 freeze_backbone=True, bidirectional=True):
        super().__init__()

        # CNN backbone: ResNet18, ImageNet-pretrained, used as a per-frame
        # feature extractor. Swap for your YOLO backbone later if desired —
        # ResNet18 here keeps this cell self-contained and easy to debug first.
        backbone = tv_models.resnet18(weights=tv_models.ResNet18_Weights.DEFAULT)
        self.feature_dim = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone

        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

        self.lstm = nn.LSTM(
            input_size=self.feature_dim,
            hidden_size=hidden_size,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=bidirectional,
        )

        lstm_out_dim = hidden_size * (2 if bidirectional else 1)
        self.classifier = nn.Sequential(
            nn.Linear(lstm_out_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        # x: (B, T, C, H, W)
        B, T_, C, H, W = x.shape
        x = x.view(B * T_, C, H, W)
        feats = self.backbone(x)                # (B*T, feature_dim)
        feats = feats.view(B, T_, self.feature_dim)  # (B, T, feature_dim)

        lstm_out, (h_n, c_n) = self.lstm(feats)
        # Use the last time step's output (both directions if bidirectional)
        seq_repr = lstm_out[:, -1, :]            # (B, lstm_out_dim)

        return self.classifier(seq_repr)

DEVICE_SEQ = DEVICE if 'DEVICE' in locals() else ('cuda' if torch.cuda.is_available() else 'cpu')

seq_model = CNNSequenceClassifier(
    num_classes=NUM_CLASSES,
    hidden_size=256,
    lstm_layers=1,
    freeze_backbone=True,   # start frozen; unfreeze later for fine-tuning if needed
    bidirectional=True,
).to(DEVICE_SEQ)

print(seq_model)

In [ ]:
# ─── Experiment 2, Step 4: Train the Sequence Classifier ───────────────────

import torch.optim as optim
from tqdm.auto import tqdm

optimizer_seq = optim.AdamW(
    filter(lambda p: p.requires_grad, seq_model.parameters()),
    lr=1e-3, weight_decay=1e-4,
)
criterion_seq = nn.CrossEntropyLoss()
scheduler_seq = optim.lr_scheduler.CosineAnnealingLR(optimizer_seq, T_max=20)

EPOCHS_SEQ = 20

def run_epoch(model, loader, optimizer, criterion, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0

    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for seqs, labels in tqdm(loader, leave=False):
            seqs, labels = seqs.to(DEVICE_SEQ), labels.to(DEVICE_SEQ)

            if train:
                optimizer.zero_grad()
            outputs = model(seqs)
            loss = criterion(outputs, labels)

            if train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * seqs.size(0)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += seqs.size(0)

    return total_loss / total, correct / total

best_val_acc = 0.0
for epoch in range(1, EPOCHS_SEQ + 1):
    train_loss, train_acc = run_epoch(seq_model, train_seq_loader, optimizer_seq, criterion_seq, train=True)
    val_loss, val_acc = run_epoch(seq_model, val_seq_loader, optimizer_seq, criterion_seq, train=False)
    scheduler_seq.step()

    print(f"Epoch {epoch:2d}/{EPOCHS_SEQ} | "
          f"Train loss {train_loss:.4f} acc {train_acc:.4f} | "
          f"Val loss {val_loss:.4f} acc {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(seq_model.state_dict(), str(RESULTS / 'seq_model_best.pt'))
        print(f"  ✅ New best val acc {val_acc:.4f} — checkpoint saved.")

print(f"\nBest validation accuracy: {best_val_acc:.4f}")

In [ ]:
# ─── Experiment 2, Step 5: Test-Set Evaluation ──────────────────────────────

seq_model.load_state_dict(torch.load(str(RESULTS / 'seq_model_best.pt')))
test_loss, test_acc = run_epoch(seq_model, test_seq_loader, optimizer_seq, criterion_seq, train=False)
print(f"Sequence model — Held-out test accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")